# **((Data Cleaning))**

## Objectives

* "We will check all the images and make sure we dont have any data represented as text or any corrupted files"
* "We will also combine all the images into one file and then split them all into our train, validation, and test sets."

## Inputs

* inputs/skin_cancer_dataset/melanoma_cancer_dataset/train
* inputs/skin_cancer_dataset/melanoma_cancer_dataset/validation
* inputs/skin_cancer_dataset/melanoma_cancer_dataset/test

## Outputs

No files will be placed in outputs, however, we will be forming the train, validation, and test folders under inputs/skin_cancer_dataset/melanoma_cancer_dataset

## Additional Comments

No additional comments

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [2]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/skin-lesion-detector/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [3]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'/workspaces/skin-lesion-detector'

---

# Data Cleaning

## Excluding Files based on their extension

Here, we will check all the images and make sure we don't have any data represented as text

In [4]:
from pathlib import Path
def clean_skin_cancer_dataset(my_data_dir):
    allowed_extensions = {'.png', '.jpeg', '.jpg'}
    class_names = ['benign', 'malignant']
    results = {}

    for class_name in class_names:
        class_dir = Path(my_data_dir) / class_name

        if not class_dir.exists():
            results[class_names] = {'status': 'folder not found', 'kept': 0, 'removed': 0}
            continue

        kept = 0
        removed = 0

        for file_path in class_dir.rglob('*'):
            if file_path.is_file():
                if file_path.suffix.lower() in allowed_extensions:
                    kept += 1
                else:
                    file_path.unlink()
                    removed += 1

        results[class_name] = {'status': 'cleaned', 'kept': kept, 'removed': removed}

    return results

train_result = clean_skin_cancer_dataset('inputs/skin_cancer_dataset/melanoma_cancer_dataset/train')
test_result  = clean_skin_cancer_dataset('inputs/skin_cancer_dataset/melanoma_cancer_dataset/test')

all_results = {'train': train_result, 'test': test_result}
print(all_results)


{'train': {'benign': {'status': 'cleaned', 'kept': 5000, 'removed': 0}, 'malignant': {'status': 'cleaned', 'kept': 4605, 'removed': 0}}, 'test': {'benign': {'status': 'cleaned', 'kept': 500, 'removed': 0}, 'malignant': {'status': 'cleaned', 'kept': 500, 'removed': 0}}}


As we can see, there are no files without the extensions .jpeg, .jpg, .png

## Removing corrupted files

Here, we will be removing images that are corrupted or cannot be opened

In [5]:
from pathlib import Path
from PIL import Image, UnidentifiedImageError


def remove_corrupt_images(data_dir):
    class_names = ['benign', 'malignant']
    results = {}

    for class_name in class_names:
        class_dir = Path(data_dir) / class_name
        removed_files = []

        if not class_dir.exists():
            results[class_name] = {
                'status': 'folder not found',
                'removed': removed_files,
            }
            continue

        for file_path in class_dir.iterdir():
            if not file_path.is_file():
                continue

            try:
                with Image.open(file_path) as image:
                    image.verify()
            except (UnidentifiedImageError, OSError, SyntaxError):
                file_path.unlink()
                removed_files.append(str(file_path))

        results[class_name] = {
            'status': 'cleaned',
            'removed': removed_files,
        }

    return results


corrupt_train_set_result = clean_skin_cancer_dataset('inputs/skin_cancer_dataset/melanoma_cancer_dataset/train')
corrupt_test_set_result  = clean_skin_cancer_dataset('inputs/skin_cancer_dataset/melanoma_cancer_dataset/test')

all_corrupt_results = {'train': corrupt_train_set_result, 'test': corrupt_test_set_result}
print(all_corrupt_results)


{'train': {'benign': {'status': 'cleaned', 'kept': 5000, 'removed': 0}, 'malignant': {'status': 'cleaned', 'kept': 4605, 'removed': 0}}, 'test': {'benign': {'status': 'cleaned', 'kept': 500, 'removed': 0}, 'malignant': {'status': 'cleaned', 'kept': 500, 'removed': 0}}}


As we can see, there were no corrupt files either

# Splitting dataset into train, validation, and test sets

Since we halready have the test and train sets, I would like to move them all into one folder so I can make sure all images are split correctly into their respective sets

In [5]:
os.makedirs('inputs/skin_cancer_dataset/melanoma_cancer_dataset/validation', exist_ok=True)
os.makedirs('inputs/skin_cancer_dataset/melanoma_cancer_dataset/validation/benign', exist_ok=True)
os.makedirs('inputs/skin_cancer_dataset/melanoma_cancer_dataset/validation/malignant', exist_ok=True)
os.makedirs('inputs/skin_cancer_dataset/melanoma_cancer_dataset/placeholder', exist_ok=True)

Now, we will combine the folders with the same label(or class) and put them in the placeholder folder for proper distribution

In [ ]:
from pathlib import Path
import shutil

base_dir = Path('inputs/skin_cancer_dataset/melanoma_cancer_dataset')
source_folders = ('train', 'test')
classes = ('benign', 'malignant')
placeholder_dir = base_dir / 'placeholder'

for class_name in classes:
    destination_dir = placeholder_dir / class_name
    destination_dir.mkdir(parents=True, exist_ok=True)

    for source_folder in source_folders:
        source_dir = base_dir / source_folder / class_name
        if not source_dir.exists():
            continue

        for source_file in source_dir.iterdir():
            if not source_file.is_file():
                continue

            destination_file = destination_dir / source_file.name
            if destination_file.exists():
                stem = source_file.stem
                suffix = source_file.suffix
                counter = 1
                while destination_file.exists():
                    destination_file = destination_dir / f'{stem}_{counter}{suffix}'
                    counter += 1

            shutil.move(str(source_file), str(destination_file))

print(f'Combined train and test images in {placeholder_dir}')

Combined train and test images in inputs/skin_cancer_dataset/melanoma_cancer_dataset/placeholder


Now, we will separate all the images into three parts. Training, validation, and test sets

In [6]:
from pathlib import Path
import random
import shutil

def split_train_validation_test_set_images(my_data_dir,
                                           train_set_ratio=0.7,
                                           validation_set_ratio=0.1,
                                           test_set_ratio=0.2):
    """
    Move images from the 'placeholder' subdirectory into train/validation/test
    splits, preserving class labels.

    The 'placeholder' directory must contain class subfolders (e.g., benign, malignant).
    The function will move all images from those subfolders into the corresponding
    class subfolders under train/, validation/, and test/.

    Args:
        my_data_dir (str or Path): Path to the dataset root (e.g.,
            'inputs/skin_cancer_dataset/melanoma_cancer_dataset').
        train_set_ratio (float): Proportion of images to assign to training.
        validation_set_ratio (float): Proportion for validation.
        test_set_ratio (float): Proportion for testing.

    Raises:
        ValueError: If the ratios do not sum to 1.0.
    """
    # Validate ratios
    total = train_set_ratio + validation_set_ratio + test_set_ratio
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f"Ratios must sum to 1.0, got {total:.3f}")

    base = Path(my_data_dir)
    placeholder_dir = base / 'placeholder'
    splits = {
        'train': base / 'train',
        'validation': base / 'validation',
        'test': base / 'test'
    }

    # Ensure destination split directories exist
    for split_path in splits.values():
        for class_name in ('benign', 'malignant'):
            (split_path / class_name).mkdir(parents=True, exist_ok=True)

    # Process each class independently
    for class_name in ('benign', 'malignant'):
        src_class_dir = placeholder_dir / class_name
        if not src_class_dir.exists():
            print(f"Warning: {src_class_dir} does not exist, skipping class {class_name}")
            continue

        # Gather all image files (any extension)
        image_files = [f for f in src_class_dir.iterdir() if f.is_file()]
        if not image_files:
            print(f"No files found in {src_class_dir}, skipping")
            continue

        # Shuffle for random distribution
        random.shuffle(image_files)

        # Compute split indices
        n = len(image_files)
        train_end = int(train_set_ratio * n)
        val_end = train_end + int(validation_set_ratio * n)

        # Partition the list
        train_files = image_files[:train_end]
        val_files = image_files[train_end:val_end]
        test_files = image_files[val_end:]

        # Move files to their respective destinations
        for split_name, file_list in zip(('train', 'validation', 'test'),
                                         (train_files, val_files, test_files)):
            dest_class_dir = splits[split_name] / class_name
            for src_file in file_list:
                dest_file = dest_class_dir / src_file.name
                # Handle possible name collisions (e.g., if files were already present)
                if dest_file.exists():
                    stem = src_file.stem
                    suffix = src_file.suffix
                    counter = 1
                    while dest_file.exists():
                        dest_file = dest_class_dir / f'{stem}_{counter}{suffix}'
                        counter += 1
                shutil.move(str(src_file), str(dest_file))

        # After moving, the class subfolder in placeholder will be empty;
        # you can optionally remove it with: src_class_dir.rmdir()

    print(f"Split complete. Images distributed to train/validation/test "
          f"with ratios {train_set_ratio:.1%}, {validation_set_ratio:.1%}, {test_set_ratio:.1%}.")

This code was 100% made by AI. I spent hours trying to do this and could not

In [7]:
split_train_validation_test_set_images(my_data_dir = 'inputs/skin_cancer_dataset/melanoma_cancer_dataset',
                                       train_set_ratio=0.7,
                                       validation_set_ratio=0.1,
                                       test_set_ratio=0.2,
                                        )

Split complete. Images distributed to train/validation/test with ratios 70.0%, 10.0%, 20.0%.


In [11]:
# remove the placeholder directory after splitting
placeholder_dir = Path('inputs/skin_cancer_dataset/melanoma_cancer_dataset/placeholder')
if placeholder_dir.exists() and placeholder_dir.is_dir():
    shutil.rmtree(placeholder_dir)
    print(f"Removed placeholder directory: {placeholder_dir}")

Removed placeholder directory: inputs/skin_cancer_dataset/melanoma_cancer_dataset/placeholder


# Conclusions and Next Steps in data cleaning
We have now cleaned the data and divided the train, validation, and test sets successfully(and after a lot of problems)

Next, we feature engineer the data so the model has an easier time learning

Also, we didnt really have any outputs to show in the dashboard, so we still do not have any outputs

Lastly, do the following commands int he terminal to save the files

git add .

git commit -m "add the message you want to commit"

git push